In [1]:
import sys, os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IS_KAGGLE:
    # Install packages not available on Kaggle
    %pip install -q kymatio kornia
    
    # Add repo to path (UPDATE 'deep-learning-course-project' to your dataset slug)
    repo_path = Path('/kaggle/input/deep-learning-course-project')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
else:
    # Local: add project root to path (assumes notebook is in notebooks/)
    project_root = Path.cwd().parent
    if (project_root / 'src').exists():
        sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import *
from src.models.architectures.RestNet18 import *
from src.models.architectures.ScatNet18 import *
from src.utils.training import *
from src.utils.visualization import *

Setting seed to 42


In [3]:
DEBUG = True
SKIP_TRAINING = True
EXP_NAME = "baseline_100"
print(f"Starting experiment {EXP_NAME}. DEBUG={DEBUG}, SKIP_TRAINING={SKIP_TRAINING}")

device = DEVICE

Starting experiment baseline_100. DEBUG=True, SKIP_TRAINING=True


In [ ]:
resnet = MakeResNet18().to(device)
MODEL_NAME = ""

total_params, model_size_mb = get_model_summary(resnet)
print(f"Total Parameters: {total_params:,}")
print(f"Model Size: {model_size_mb:.2f} MB")

Total Parameters: 11,173,962
Model Size: 42.63 MB


In [5]:
trainloader, valloader, testloader, train_set, val_set, test_set = get_cifar10_loaders_and_splits(n_samples_per_class_train=100)
resnet_optimizer, resnet_scheduler = get_optimizer_and_scheduler(resnet)

Files already downloaded and verified
Using default 500 samples per class for val.
Files already downloaded and verified
Original train-val size: 50000
Train size: 1000
Val size: 5000
Test size: 10000
Samples per class (train): Counter({np.int64(7): 100, np.int64(1): 100, np.int64(8): 100, np.int64(4): 100, np.int64(3): 100, np.int64(9): 100, np.int64(5): 100, np.int64(0): 100, np.int64(2): 100, np.int64(6): 100})
Samples per class (val): Counter({np.int64(1): 500, np.int64(9): 500, np.int64(2): 500, np.int64(4): 500, np.int64(6): 500, np.int64(5): 500, np.int64(0): 500, np.int64(3): 500, np.int64(8): 500, np.int64(7): 500})


In [ ]:
if not SKIP_TRAINING:
    train_model(
        model=resnet,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=resnet_optimizer,
        scheduler=resnet_scheduler,
        device=device,
        experiment_name=EXP_NAME,
        model_name=MODEL_NAME,
        DEBUG=DEBUG
    )

In [ ]:
debug_suff = "_DEBUG" if DEBUG else ""
load_weights(resnet, experiment_name=EXP_NAME, model_name=(MODEL_NAME+ debug_suff), device=device)
if not SKIP_TRAINING:
    print(f'Final test accuracy is: {calculate_accuracy(resnet, testloader, device):.3f}')

Final test accuracy is: 46.160
